# Dylan Wendling 
# Final project
# A 2D Artillery Game featuring dynamic terrain, maps, phasing projectiles, and a fully functional pause menu.

In [ ]:
# File: pyshock.py
# A 2D Artillery Game featuring bedrock limits and an earthquake terrain-regeneration system.

import pygame
import math
import random
import sys

# --- CONFIGURATION & CONSTANTS ---
WIDTH, HEIGHT = 800, 600
FPS = 60

# Colors
SKY_BLUE = (135, 206, 235)
CLOUD_COLOR = (255, 255, 255, 180)
MOUNTAIN_COLOR = (100, 110, 130)
TERRAIN_COLOR = (34, 139, 34)
CRATER_COLOR = (139, 69, 19)
BEDROCK_COLOR = (40, 40, 40)
P1_COLOR = (50, 50, 255)
P2_COLOR = (255, 50, 50)
UI_TEXT = (255, 255, 255)
CRATE_COLOR = (200, 150, 50)

# Physics & Limits
GRAVITY = 0.2
MAX_POWER = 25
EXPLOSION_RADIUS = 40
DIRT_BOMB_RADIUS = 50
MAX_HEALTH = 100
MAX_FUEL = 100
MOVE_SPEED = 2
BEDROCK_Y = HEIGHT - 40 # The unbreakable floor

class Background:
    def __init__(self):
        self.clouds = []
        self.mountains = []
        self.generate_mountains()
        self.generate_clouds()

    def generate_mountains(self):
        self.mountains = [(0, BEDROCK_Y)]
        for x in range(0, WIDTH + 100, 100):
            self.mountains.append((x, random.randint(HEIGHT // 3, HEIGHT // 2 + 50)))
        self.mountains.append((WIDTH, BEDROCK_Y))

    def generate_clouds(self):
        for _ in range(8):
            self.clouds.append([
                random.randint(0, WIDTH), random.randint(20, 150), 
                random.randint(20, 50), random.uniform(0.1, 0.4)
            ])

    def update_and_draw(self, surface):
        pygame.draw.polygon(surface, MOUNTAIN_COLOR, self.mountains)
        for cloud in self.clouds:
            cloud[0] += cloud[3] 
            if cloud[0] - cloud[2] > WIDTH:
                cloud[0] = -cloud[2]
            pygame.draw.circle(surface, CLOUD_COLOR, (int(cloud[0]), int(cloud[1])), int(cloud[2]))
            pygame.draw.circle(surface, CLOUD_COLOR, (int(cloud[0] + cloud[2]*0.8), int(cloud[1] + cloud[2]*0.2)), int(cloud[2]*0.8))
            pygame.draw.circle(surface, CLOUD_COLOR, (int(cloud[0] - cloud[2]*0.6), int(cloud[1] + cloud[2]*0.3)), int(cloud[2]*0.7))

class Terrain:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.heightmap = []
        self.generate()

    def generate(self):
        self.heightmap = []
        base_height = self.height * 0.65
        offset1 = random.uniform(0, 2 * math.pi)
        offset2 = random.uniform(0, 2 * math.pi)
        for x in range(self.width):
            h = base_height + math.sin((x / 150) + offset1) * 60 + math.sin((x / 50) + offset2) * 20
            # Ensure natural generation doesn't dip below bedrock
            self.heightmap.append(min(int(h), BEDROCK_Y))

    def draw(self, surface):
        # Draw dynamic dirt/grass lines
        for x in range(self.width):
            y = self.heightmap[x]
            pygame.draw.line(surface, TERRAIN_COLOR, (x, y), (x, BEDROCK_Y), 1)
            pygame.draw.line(surface, CRATER_COLOR, (x, y), (x, min(y+5, BEDROCK_Y)), 1)
        
        # Draw solid bedrock at the bottom
        pygame.draw.rect(surface, BEDROCK_COLOR, (0, BEDROCK_Y, self.width, self.height - BEDROCK_Y))
        pygame.draw.line(surface, (20, 20, 20), (0, BEDROCK_Y), (self.width, BEDROCK_Y), 3)

    def modify_terrain(self, cx, cy, radius, is_dirt_bomb=False):
        cx, cy = int(cx), int(cy)
        for x in range(max(0, cx - radius), min(self.width, cx + radius)):
            dx = x - cx
            dy = math.sqrt(max(0, radius**2 - dx**2))
            if is_dirt_bomb:
                mound_top = cy - dy
                if self.heightmap[x] > mound_top: 
                    self.heightmap[x] = int(mound_top)
            else:
                explosion_bottom = cy + dy
                # Clamp the explosion depth to the BEDROCK_Y limit
                clamped_bottom = min(int(explosion_bottom), BEDROCK_Y)
                if self.heightmap[x] < clamped_bottom: 
                    self.heightmap[x] = clamped_bottom

class SupplyDrop:
    def __init__(self, x):
        self.x = x
        self.y = -50
        self.width = 20
        self.height = 20
        self.type = random.choice(["Armor", "Health"])
        self.color = (150, 150, 150) if self.type == "Armor" else (50, 200, 50)

    def update(self, terrain):
        target_y = terrain.heightmap[int(self.x)]
        if self.y < target_y - self.height:
            self.y += 2
        else:
            self.y = target_y - self.height

    def draw(self, surface):
        rect = pygame.Rect(self.x - self.width//2, self.y, self.width, self.height)
        pygame.draw.rect(surface, CRATE_COLOR, rect)
        
        if self.type == "Armor":
            pygame.draw.polygon(surface, self.color, [(self.x, self.y+2), (self.x-6, self.y+8), (self.x, self.y+16), (self.x+6, self.y+8)])
        else:
            pygame.draw.rect(surface, self.color, (self.x-2, self.y+4, 4, 12))
            pygame.draw.rect(surface, self.color, (self.x-6, self.y+8, 12, 4))
            
        target_y = game.terrain.heightmap[int(self.x)]
        if self.y < target_y - self.height - 5:
            pygame.draw.arc(surface, (255,255,255), (self.x-15, self.y-20, 30, 30), 0, math.pi, 2)
            pygame.draw.line(surface, (200,200,200), (self.x-15, self.y-5), (self.x-5, self.y))
            pygame.draw.line(surface, (200,200,200), (self.x+15, self.y-5), (self.x+5, self.y))

class Tank:
    def __init__(self, x, color, name):
        self.x = x
        self.y = 0
        self.color = color
        self.name = name
        self.health = MAX_HEALTH
        self.armor = 0
        self.fuel = MAX_FUEL
        self.angle = 45 if x < WIDTH/2 else 135
        self.power = 15
        self.width = 20
        self.height = 10
        self.weapons = ["Standard", "Cluster", "Dirt Bomb"]
        self.weapon_idx = 0

    def get_current_weapon(self):
        return self.weapons[self.weapon_idx]

    def cycle_weapon(self):
        self.weapon_idx = (self.weapon_idx + 1) % len(self.weapons)

    def move(self, direction, terrain):
        if self.fuel > 0:
            self.x += direction * MOVE_SPEED
            self.x = max(self.width, min(WIDTH - self.width, self.x))
            self.fuel -= 1
            self.update_position(terrain)

    def update_position(self, terrain):
        self.y = terrain.heightmap[int(self.x)]

    def take_damage(self, amount):
        if self.armor > 0:
            if amount > self.armor:
                amount -= self.armor
                self.armor = 0
                self.health -= amount
            else:
                self.armor -= amount
        else:
            self.health -= amount
        self.health = max(0, self.health)

    def draw_aim_assist(self, surface, wind):
        px, py = self.x, self.y - self.height
        rad = math.radians(self.angle)
        vx = math.cos(rad) * self.power
        vy = -math.sin(rad) * self.power
        
        for _ in range(18):
            vy += GRAVITY
            vx += wind
            px += vx
            py += vy
            if 0 <= px < WIDTH and py < game.terrain.heightmap[int(px)]:
                pygame.draw.circle(surface, (255, 255, 255, 100), (int(px), int(py)), 2)

    def draw(self, surface):
        if self.health <= 0: return
        rect = pygame.Rect(self.x - self.width//2, self.y - self.height, self.width, self.height)
        pygame.draw.rect(surface, self.color, rect)
        pygame.draw.circle(surface, self.color, (int(self.x), int(self.y - self.height)), self.width//2)

        rad = math.radians(self.angle)
        end_x = self.x + math.cos(rad) * 20
        end_y = self.y - self.height - math.sin(rad) * 20
        pygame.draw.line(surface, (0, 0, 0), (self.x, self.y - self.height), (end_x, end_y), 4)
        
        fuel_pct = self.fuel / MAX_FUEL
        pygame.draw.rect(surface, (50,50,50), (self.x - 15, self.y - 30, 30, 4))
        pygame.draw.rect(surface, (255,255,0), (self.x - 15, self.y - 30, 30 * fuel_pct, 4))

class Projectile:
    def __init__(self, x, y, angle, power, wind, weapon_type):
        self.x = x
        self.y = y
        rad = math.radians(angle)
        self.vx = math.cos(rad) * power
        self.vy = -math.sin(rad) * power
        self.wind = wind
        self.weapon_type = weapon_type
        self.trail = []

    def update(self):
        self.trail.append((int(self.x), int(self.y)))
        if len(self.trail) > 15: self.trail.pop(0)

        self.vy += GRAVITY
        self.vx += self.wind
        self.x += self.vx
        self.y += self.vy

    def draw(self, surface):
        color = (139, 69, 19) if self.weapon_type == "Dirt Bomb" else (50, 50, 50)
        if len(self.trail) > 1:
            pygame.draw.lines(surface, (200, 200, 200), False, self.trail, 2)
        pygame.draw.circle(surface, color, (int(self.x), int(self.y)), 4)

class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        # Internal surface for screen shake effects
        self.display = pygame.Surface((WIDTH, HEIGHT)) 
        pygame.display.set_caption("PyShock - Dynamic Terrain")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("Arial", 16, bold=True)
        self.big_font = pygame.font.SysFont("Arial", 36, bold=True)
        self.huge_font = pygame.font.SysFont("Arial", 60, bold=True)
        
        self.background = Background()
        self.terrain = Terrain(WIDTH, HEIGHT)
        self.p1 = Tank(100, P1_COLOR, "Player 1")
        self.p2 = Tank(WIDTH - 100, P2_COLOR, "Player 2")
        self.wind = random.uniform(-0.1, 0.1)
        
        self.active_player = self.p1
        self.projectiles = []
        self.explosions = [] 
        self.supplies = []
        
        self.state = "AIMING" 
        self.total_turns = 0
        self.quake_timer = 0
        
        self.p1.update_position(self.terrain)
        self.p2.update_position(self.terrain)

    def handle_input(self):
        keys = pygame.key.get_pressed()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
                
            if event.type == pygame.KEYDOWN:
                if self.state == "AIMING":
                    if event.key == pygame.K_SPACE:
                        self.fire_projectile()
                    if event.key == pygame.K_TAB:
                        self.active_player.cycle_weapon()
                elif self.state == "GAMEOVER" and event.key == pygame.K_RETURN:
                    self.reset_round()

        if self.state == "AIMING":
            if keys[pygame.K_a]: self.active_player.move(-1, self.terrain)
            if keys[pygame.K_d]: self.active_player.move(1, self.terrain)
            if keys[pygame.K_LEFT]: self.active_player.angle = min(180, self.active_player.angle + 1)
            if keys[pygame.K_RIGHT]: self.active_player.angle = max(0, self.active_player.angle - 1)
            if keys[pygame.K_UP]: self.active_player.power = min(MAX_POWER, self.active_player.power + 0.2)
            if keys[pygame.K_DOWN]: self.active_player.power = max(1, self.active_player.power - 0.2)

    def fire_projectile(self):
        start_x = self.active_player.x
        start_y = self.active_player.y - self.active_player.height
        weapon = self.active_player.get_current_weapon()
        
        if weapon == "Cluster":
            for offset in [-5, 0, 5]:
                p = Projectile(start_x, start_y, self.active_player.angle + offset, self.active_player.power, self.wind, weapon)
                self.projectiles.append(p)
        else:
            p = Projectile(start_x, start_y, self.active_player.angle, self.active_player.power, self.wind, weapon)
            self.projectiles.append(p)
            
        self.state = "FIRING"

    def process_projectiles(self):
        for p in self.projectiles[:]:
            p.update()
            if p.x < -100 or p.x > WIDTH + 100 or p.y > HEIGHT + 100:
                self.projectiles.remove(p)
                continue

            px, py = int(p.x), int(p.y)
            if 0 <= px < WIDTH and py >= self.terrain.heightmap[px]:
                self.projectiles.remove(p)
                is_dirt = (p.weapon_type == "Dirt Bomb")
                radius = DIRT_BOMB_RADIUS if is_dirt else EXPLOSION_RADIUS
                
                self.terrain.modify_terrain(px, py, radius, is_dirt_bomb=is_dirt)
                self.explosions.append({'pos': (px, py), 'timer': 30, 'type': p.weapon_type})
                
                for crate in self.supplies[:]:
                    if math.hypot(crate.x - px, crate.y - py) < radius:
                        self.supplies.remove(crate)

                if not is_dirt:
                    self.damage_tanks(px, py, radius)

    def process_explosions(self):
        for exp in self.explosions[:]:
            exp['timer'] -= 1
            if exp['timer'] <= 0:
                self.explosions.remove(exp)

    def damage_tanks(self, cx, cy, radius):
        for tank in [self.p1, self.p2]:
            if tank.health <= 0: continue
            dist = math.hypot(tank.x - cx, tank.y - cy)
            if dist < radius + 10:
                damage = int((1 - (dist / (radius + 10))) * 40)
                tank.take_damage(damage)
                if tank.health <= 0:
                    self.state = "GAMEOVER"

    def check_supply_collection(self):
        for crate in self.supplies[:]:
            dist = math.hypot(self.active_player.x - crate.x, self.active_player.y - crate.y)
            if dist < 25:
                if crate.type == "Armor": self.active_player.armor += 30
                elif crate.type == "Health": self.active_player.health = min(MAX_HEALTH, self.active_player.health + 30)
                self.supplies.remove(crate)

    def next_turn(self):
        self.wind = random.uniform(-0.1, 0.1)
        self.active_player = self.p2 if self.active_player == self.p1 else self.p1
        self.active_player.fuel = MAX_FUEL 
        self.total_turns += 1
        
        # Spawn supply drop 30% chance
        if random.random() < 0.3:
            self.supplies.append(SupplyDrop(random.randint(100, WIDTH-100)))

        # Trigger Earthquake every 6 turns (3 rounds)
        if self.total_turns > 0 and self.total_turns % 6 == 0:
            self.state = "EARTHQUAKE"
            self.quake_timer = 90
        else:
            self.state = "AIMING"

    def reset_round(self):
        self.terrain.generate()
        self.supplies.clear()
        for tank in [self.p1, self.p2]:
            tank.health = MAX_HEALTH
            tank.armor = 0
            tank.fuel = MAX_FUEL
            tank.update_position(self.terrain)
        self.wind = random.uniform(-0.1, 0.1)
        self.total_turns = 0
        self.state = "AIMING"
        self.active_player = self.p1

    def draw_ui(self, surface):
        wind_dir = ">>" if self.wind > 0 else "<<"
        wind_text = self.font.render(f"Wind: {abs(self.wind*100):.1f} {wind_dir}", True, UI_TEXT)
        surface.blit(wind_text, (WIDTH//2 - 40, 10))
        
        turn_text = self.font.render(f"{self.active_player.name}'s Turn", True, self.active_player.color)
        surface.blit(turn_text, (WIDTH//2 - 50, 30))

        p1_stats = self.font.render(f"HP: {self.p1.health} | Arm: {self.p1.armor} | Pow: {self.p1.power:.1f}", True, P1_COLOR)
        p1_wep = self.font.render(f"Wep: {self.p1.get_current_weapon()}", True, P1_COLOR)
        surface.blit(p1_stats, (10, 10))
        surface.blit(p1_wep, (10, 30))

        p2_stats = self.font.render(f"HP: {self.p2.health} | Arm: {self.p2.armor} | Pow: {self.p2.power:.1f}", True, P2_COLOR)
        p2_wep = self.font.render(f"Wep: {self.p2.get_current_weapon()}", True, P2_COLOR)
        surface.blit(p2_stats, (WIDTH - 250, 10))
        surface.blit(p2_wep, (WIDTH - 250, 30))

        controls = self.font.render("Move: A/D | Aim: Arrows | Wep: TAB | Fire: SPACE", True, (200, 200, 200))
        surface.blit(controls, (WIDTH//2 - 170, HEIGHT - 25))

        if self.state == "GAMEOVER":
            winner = "Player 1" if self.p2.health <= 0 else "Player 2"
            msg = self.big_font.render(f"{winner} Wins! Press ENTER.", True, (255, 255, 0))
            surface.blit(msg, (WIDTH//2 - 180, HEIGHT//2))
            
        elif self.state == "EARTHQUAKE":
            msg = self.huge_font.render("EARTHQUAKE!", True, (255, 100, 0))
            surface.blit(msg, (WIDTH//2 - 200, HEIGHT//2 - 50))

    def update(self):
        for crate in self.supplies:
            crate.update(self.terrain)

        if self.state == "AIMING":
            self.p1.update_position(self.terrain)
            self.p2.update_position(self.terrain)
            self.check_supply_collection()

        elif self.state == "FIRING":
            self.process_projectiles()
            self.process_explosions()
            if len(self.projectiles) == 0 and len(self.explosions) == 0:
                if self.state != "GAMEOVER":
                    self.next_turn()
                    
        elif self.state == "EARTHQUAKE":
            self.quake_timer -= 1
            # Regenerate terrain halfway through the shake
            if self.quake_timer == 45:
                self.terrain.generate()
                self.p1.update_position(self.terrain)
                self.p2.update_position(self.terrain)
                for crate in self.supplies:
                    crate.update(self.terrain) # Snap to new ground
            elif self.quake_timer <= 0:
                self.state = "AIMING"

    def draw(self):
        # Draw everything to the internal display surface first
        self.display.fill(SKY_BLUE)
        self.background.update_and_draw(self.display)
        self.terrain.draw(self.display)
        
        for crate in self.supplies:
            crate.draw(self.display)

        self.p1.draw(self.display)
        self.p2.draw(self.display)
        if self.state == "AIMING":
            self.active_player.draw_aim_assist(self.display, self.wind)

        for p in self.projectiles:
            p.draw(self.display)

        for exp in self.explosions:
            color = (139, 69, 19) if exp['type'] == "Dirt Bomb" else (255, 100, 0)
            pygame.draw.circle(self.display, color, exp['pos'], EXPLOSION_RADIUS)

        self.draw_ui(self.display)
        
        # Apply screen shake if in EARTHQUAKE state by offsetting the blit
        shake_x, shake_y = 0, 0
        if self.state == "EARTHQUAKE":
            shake_x = random.randint(-10, 10)
            shake_y = random.randint(-10, 10)
            
        self.screen.blit(self.display, (shake_x, shake_y))
        pygame.display.flip()

    def run(self):
        while True:
            self.handle_input()
            self.update()
            self.draw()
            self.clock.tick(FPS)

if __name__ == "__main__":
    global game
    game = Game()
    game.run()